In [5]:

# Import pandas to work with tabular data
import pandas as pd

# Load the Car Evaluation dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"

columns = [
    "buying", "maint", "doors", "persons",
    "lug_boot", "safety", "class"
]

df = pd.read_csv(url, names=columns)

# Display the first five rows
print("First five rows of the dataset:")
display(df.head())

# Check the dataset size and target categories
print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["class"].value_counts())


First five rows of the dataset:


,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


Dataset shape: (1728, 7)

Class distribution:
class
unacc    1210
acc       384
good       69
vgood      65
Name: count, dtype: int64


In [6]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

# Separate input features from the target class
X = df.drop(columns=["class"])
y = df["class"]

# Convert categorical input features into numeric values
encoder = OrdinalEncoder()
X_encoded = pd.DataFrame(
    encoder.fit_transform(X),
    columns=X.columns
)

# Encode the four target classes
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Target classes:", list(label_encoder.classes_))


Training records: 1382
Testing records: 346
Target classes: ['acc', 'good', 'unacc', 'vgood']


In [7]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Train a decision tree with a limit on its depth
tree_model = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=3,
    random_state=42
)

tree_model.fit(X_train, y_train)

# Make predictions on the test data
tree_predictions = tree_model.predict(X_test)

# Evaluate the decision tree
print("Decision Tree Accuracy:",
      round(accuracy_score(y_test, tree_predictions), 4))

print("\nClassification Report:")
print(classification_report(
    y_test,
    tree_predictions,
    labels=range(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, tree_predictions))


Decision Tree Accuracy: 0.8931

Classification Report:
              precision    recall  f1-score   support

         acc       0.69      0.95      0.80        77
        good       0.00      0.00      0.00        14
       unacc       1.00      0.92      0.96       242
       vgood       0.76      1.00      0.87        13

    accuracy                           0.89       346
   macro avg       0.61      0.72      0.66       346
weighted avg       0.88      0.89      0.88       346

Confusion Matrix:
[[ 73   0   0   4]
 [ 14   0   0   0]
 [ 19   0 223   0]
 [  0   0   0  13]]


In [8]:

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

# Train the Random Forest classifier
rf_model = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=2,
    random_state=42
)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

# Train the Extra Trees classifier
et_model = ExtraTreesClassifier(
    n_estimators=200,
    min_samples_leaf=2,
    random_state=42
)
et_model.fit(X_train, y_train)
et_predictions = et_model.predict(X_test)

# Compare the three models
results = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest", "Extra Trees"],
    "Accuracy": [
        accuracy_score(y_test, tree_predictions),
        accuracy_score(y_test, rf_predictions),
        accuracy_score(y_test, et_predictions)
    ]
})

results["Accuracy"] = results["Accuracy"].round(4)
display(results.sort_values("Accuracy", ascending=False))


,Model,Accuracy
1,Random Forest,0.9827
2,Extra Trees,0.9682
0,Decision Tree,0.8931


### Results and discussion -  Rim -

The decision tree, Random Forest, and Extra Trees classifiers were trained using the Car Evaluation dataset. The dataset was divided into training and testing sets to evaluate how well each model classified cars into the four categories.

The models were compared using test accuracy and classification reports. The decision tree provides a model that is relatively easy to understand, while Random Forest and Extra Trees combine multiple trees to improve predictive performance. The comparison table shows which model achieved the highest accuracy on the test data.

These results can help a car rental company make more informed decisions when evaluating vehicles. However, accuracy alone does not show every type of classification error, so the confusion matrices and classification reports should also be considered.